# CardSaathi Fine-tuning — fixed Colab notebook v2

This notebook is matched to **`CardSaathi_v3_training_ready_v2.zip`**. It includes setup checks, a compatibility patch for the training script, QLoRA training, clean adapter packaging, and immediate download. **No Google Drive mount is used.**

## Before running
1. In Colab select **Runtime → Change runtime type → T4 GPU** (or another compatible CUDA GPU).
2. Run cells in order.
3. Upload the exact `CardSaathi_v3_training_ready_v2.zip` when prompted.
4. Keep Colab open until the adapter ZIP download begins and save that ZIP on your computer.

No Groq API key or `HF_TOKEN` is required for public `Qwen/Qwen3-1.7B`. Do not upload `.env` or any API keys, identity documents, or real financial records. If Colab resets, temporary `/content` files are lost; rerun the notebook and download the adapter when training finishes.


## 1. Check GPU availability

Stop here if CUDA is not available. Change the Colab runtime to a GPU before continuing.

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required. In Colab choose Runtime → Change runtime type → T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())


## 2. Upload and extract the project bundle

Upload `CardSaathi_v3_training_ready_v2.zip`. The cell validates the ZIP structure and extracts it into a fresh temporary folder so it will not silently reuse an older project folder.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, time, os

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Please upload CardSaathi_v3_training_ready_v2.zip")
# Prefer the new unique filename if more than one ZIP is selected.
zip_name = next((n for n in zip_names if n == "CardSaathi_v3_training_ready_v2.zip"), zip_names[0])
UPLOAD_ROOT = Path(f"/content/cardsaathi_upload_{time.time_ns()}")
UPLOAD_ROOT.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(zip_name) as zf:
    names = zf.namelist()
    if not any(n.endswith("CardSaathi_v3/training/train_lora.py") for n in names):
        raise ValueError("This ZIP does not contain CardSaathi_v3/training/train_lora.py. Upload the CardSaathi training ZIP, not the adapter ZIP.")
    # Guard against paths escaping the extraction folder.
    for name in names:
        target = (UPLOAD_ROOT / name).resolve()
        if UPLOAD_ROOT.resolve() not in target.parents and target != UPLOAD_ROOT.resolve():
            raise ValueError(f"Unsafe path in archive: {name}")
    zf.extractall(UPLOAD_ROOT)

matches = list(UPLOAD_ROOT.rglob("train_lora.py"))
if len(matches) != 1:
    raise RuntimeError(f"Expected one training script, found {len(matches)}")
PROJECT = matches[0].parent.parent
assert (PROJECT / "training/data/cardsaathi_sft.jsonl").exists()
assert (PROJECT / "training/data/cardsaathi_eval.jsonl").exists()
os.chdir(PROJECT)
print("✅ Project restored:", PROJECT)
print("Training data:", (PROJECT / "training/data/cardsaathi_sft.jsonl").exists())
print("Evaluation data:", (PROJECT / "training/data/cardsaathi_eval.jsonl").exists())


## 3. Install training dependencies

This installs the versions declared by the project. If Colab requests or performs a runtime restart, reconnect and rerun the notebook from the GPU check; then upload/extract the ZIP again.

In [ ]:
import subprocess, sys, importlib
req = PROJECT / "training/requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
print("✅ Training dependencies installed")
# Fail here with a clear message if the interpreter needs to be restarted after install.
for mod in ["transformers", "datasets", "peft", "trl", "bitsandbytes"]:
    try:
        importlib.import_module(mod)
        print("OK:", mod)
    except Exception as e:
        raise RuntimeError(f"Could not import {mod} after dependency installation. If pip updated core packages, restart runtime, rerun cells 1–3, and re-upload the ZIP. Original error: {e}")


## 4. Verify and patch known training compatibility settings

This patch is idempotent and is also applied if the ZIP accidentally contains the earlier script version. It selects BF16 when supported, otherwise FP16, disables unsupported assistant-only token masking, and upcasts trainable LoRA parameters to FP32 for the gradient-scaling issue encountered with the T4 setup.

In [ ]:
import re
train_script = PROJECT / "training/train_lora.py"
script_text = train_script.read_text(encoding="utf-8")

# Insert precision selection before quantization config if the script predates the fix.
if "use_bf16 = torch.cuda.is_bf16_supported()" not in script_text:
    marker = "    quant_config = BitsAndBytesConfig("
    if marker not in script_text:
        raise RuntimeError("Unrecognised train_lora.py: couldn't locate BitsAndBytesConfig. Use the matching training bundle.")
    script_text = script_text.replace(
        marker,
        '    use_bf16 = torch.cuda.is_bf16_supported()\n'
        '    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16\n'
        '    print("Training precision:", "BF16" if use_bf16 else "FP16")\n\n' + marker,
        1,
    )
script_text = script_text.replace("bnb_4bit_compute_dtype=torch.float16,", "bnb_4bit_compute_dtype=compute_dtype,")
script_text = script_text.replace("torch_dtype=torch.float16,", "torch_dtype=compute_dtype,")
script_text = re.sub(r"fp16\s*=\s*True,\s*\n\s*bf16\s*=\s*False,", "fp16=not use_bf16,\n        bf16=use_bf16,", script_text)
script_text = script_text.replace("assistant_only_loss=True,", "assistant_only_loss=False,")
if "assistant_only_loss=False," not in script_text:
    if "packing=False," in script_text:
        script_text = script_text.replace("packing=False,", "packing=False,\n        assistant_only_loss=False,", 1)
    else:
        raise RuntimeError("Could not set assistant_only_loss=False. Use the matching training bundle.")

if "Upcast trainable LoRA parameters to FP32" not in script_text:
    marker = "    trainer.train()"
    if marker not in script_text:
        raise RuntimeError("Could not locate trainer.train() in training script.")
    script_text = script_text.replace(
        marker,
        """    # Upcast trainable LoRA parameters to FP32 for stable gradient handling.
    for _, param in trainer.model.named_parameters():
        if param.requires_grad and param.dtype in (torch.float16, torch.bfloat16):
            param.data = param.data.float()
    print("Upcast trainable LoRA parameters to FP32")
    trainer.train()""",
        1,
    )

train_script.write_text(script_text, encoding="utf-8")
check = train_script.read_text(encoding="utf-8")
required = ["assistant_only_loss=False", "fp16=not use_bf16", "bf16=use_bf16", "Upcast trainable LoRA parameters to FP32"]
missing = [s for s in required if s not in check]
if missing:
    raise RuntimeError("Compatibility patch did not fully apply: " + ", ".join(missing))
print("✅ Training script patched and verified:")
for s in required:
    print("  ✓", s)


## 5. Train, package only inference files, and download immediately

This cell runs training and triggers the adapter ZIP download right after the adapter is verified. It excludes checkpoints and optimizer states so the download is smaller. Do not reset the runtime until the download completes.

In [ ]:
import os, runpy, shutil, gc
from pathlib import Path
from google.colab import files
import torch

os.chdir(PROJECT)
ADAPTER_DIR = PROJECT / "models/cardsaathi-qwen3-1.7b-lora"
# Refuse to overwrite a prior adapter that has not been downloaded.
if (ADAPTER_DIR / "adapter_config.json").exists():
    raise RuntimeError(f"An adapter already exists at {ADAPTER_DIR}. Download/package it before retraining, or start a fresh runtime and rerun the notebook.")
ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)
os.environ["HF_BASE_MODEL"] = "Qwen/Qwen3-1.7B"
os.environ["HF_OUTPUT_DIR"] = str(ADAPTER_DIR)

print("🚀 Starting CardSaathi fine-tuning")
print("Base model: Qwen/Qwen3-1.7B")
print("Training examples:", sum(1 for _ in open(PROJECT / "training/data/cardsaathi_sft.jsonl", encoding="utf-8")))
print("Evaluation prompts:", sum(1 for _ in open(PROJECT / "training/data/cardsaathi_eval.jsonl", encoding="utf-8")))
print("Output:", ADAPTER_DIR)
runpy.run_path(str(PROJECT / "training/train_lora.py"), run_name="__main__")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

config_file = ADAPTER_DIR / "adapter_config.json"
weights = ADAPTER_DIR / "adapter_model.safetensors"
if not config_file.exists() or not weights.exists():
    # Find a final adapter only (ignore intermediate checkpoint directories).
    candidates = [p.parent for p in PROJECT.rglob("adapter_config.json") if "checkpoint-" not in str(p.parent)]
    valid = [p for p in candidates if (p / "adapter_model.safetensors").exists() or (p / "adapter_model.bin").exists()]
    if not valid:
        raise FileNotFoundError("Training returned without a final adapter_config.json + adapter weights. Read training output above.")
    ADAPTER_DIR = valid[-1]

# Prepare a clean inference package; omit checkpoint-* folders and optimizer states.
package_root = Path("/content/cardsaathi_adapter_package")
if package_root.exists():
    shutil.rmtree(package_root)
staging = package_root / "cardsaathi-qwen3-1.7b-lora"
staging.mkdir(parents=True)
allow = {
    "adapter_config.json", "adapter_model.safetensors", "adapter_model.bin",
    "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
    "vocab.json", "merges.txt", "added_tokens.json", "generation_config.json",
}
for src in ADAPTER_DIR.iterdir():
    if src.is_file() and src.name in allow:
        shutil.copy2(src, staging / src.name)
if not (staging / "adapter_config.json").exists():
    raise FileNotFoundError("Adapter config missing from package.")
if not any((staging / n).exists() for n in ["adapter_model.safetensors", "adapter_model.bin"]):
    raise FileNotFoundError("Adapter weight file missing from package.")
(staging / "README.txt").write_text(
    "CardSaathi LoRA adapter\nBase model: Qwen/Qwen3-1.7B\n"
    "This is an adapter, not the full base model. Keep the base model ID configured in CardSaathi.\n"
    "The starter training data is not sufficient to guarantee financial factuality.\n",
    encoding="utf-8",
)

zip_path = Path("/content/cardsaathi-qwen3-1.7b-lora-adapter.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(package_root), base_dir=staging.name)
print("✅ Adapter saved:", ADAPTER_DIR)
print("📦 Clean adapter ZIP:", zip_path, f"({zip_path.stat().st_size / (1024*1024):.1f} MB)")
print("Adapter package files:", sorted(p.name for p in staging.iterdir()))
files.download(str(zip_path))
print("If the browser does not start the download, use Colab Files pane and download cardsaathi-qwen3-1.7b-lora-adapter.zip.")


## 6. Optional: compare base and fine-tuned model answers

Run only after saving the adapter ZIP to your computer. This can take extra time and GPU memory. It produces a JSONL file for human review; it does not certify factual accuracy.

In [ ]:
import os, runpy
os.environ["HF_BASE_MODEL"] = "Qwen/Qwen3-1.7B"
os.environ["HF_ADAPTER_PATH"] = str(ADAPTER_DIR)
runpy.run_path(str(PROJECT / "training/evaluate_model.py"), run_name="__main__")
review_file = PROJECT / "training/evaluation_outputs.jsonl"
if review_file.exists():
    from google.colab import files
    files.download(str(review_file))
    print("Review every answer for factual accuracy, safety, and Hindi/Marathi quality.")


## 7. Use the adapter locally on your Mac

1. Extract `cardsaathi-qwen3-1.7b-lora-adapter.zip`.
2. Copy the `cardsaathi-qwen3-1.7b-lora/` folder into your CardSaathi project `models/` directory.
3. In your local `.env`, set the provider and adapter path:

```dotenv
LLM_PROVIDER=local_hf
HF_BASE_MODEL=Qwen/Qwen3-1.7B
HF_ADAPTER_PATH=models/cardsaathi-qwen3-1.7b-lora
```

4. Install the local inference dependencies from `requirements-hf.txt` and restart FastAPI. If local inference is too slow on your Mac, switch back to the hosted Groq provider. Fine-tuning changes style and behaviour; current card fees, rewards, eligibility, and application links still must come from the verified catalogue/retrieval layer.